<a href="https://colab.research.google.com/github/AnanyaAsthana/Hadoop-CUDA-Lab/blob/main/CUDAlab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi


Wed Feb  4 09:44:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile matrix_add.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <time.h>

__global__ void matrixAdd(float *A, float *B, float *C, int N)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N)
    {
        int idx = row * N + col;
        C[idx] = A[idx] + B[idx];
    }
}

int main()
{
    int N;
    float valA = 5.0f, valB = 7.0f;

    printf("Enter matrix size (N): ");
    scanf("%d", &N);

    int size = N * N * sizeof(float);

    float *h_A = (float *)malloc(size);
    float *h_B = (float *)malloc(size);
    float *h_C_cpu = (float *)malloc(size);
    float *h_C_gpu = (float *)malloc(size);

    for (int i = 0; i < N * N; i++)
    {
        h_A[i] = valA;
        h_B[i] = valB;
    }

    clock_t cpu_start = clock();
    for (int i = 0; i < N * N; i++)
        h_C_cpu[i] = h_A[i] + h_B[i];
    clock_t cpu_end = clock();

    float cpu_time = 1000.0f * (cpu_end - cpu_start) / CLOCKS_PER_SEC;

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dim3 block(16, 16);
    dim3 grid((N + block.x - 1) / block.x,
              (N + block.y - 1) / block.y);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    matrixAdd<<<grid, block>>>(d_A, d_B, d_C, N);
    cudaEventRecord(stop);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
    {
        printf("CUDA error: %s\n", cudaGetErrorString(err));
        return 1;
    }

    cudaEventSynchronize(stop);

    float gpu_time;
    cudaEventElapsedTime(&gpu_time, start, stop);

    cudaMemcpy(h_C_gpu, d_C, size, cudaMemcpyDeviceToHost);

    printf("\nCPU Time (ms): %.4f\n", cpu_time);
    printf("GPU Kernel Time (ms): %.4f\n", gpu_time);

    if (N <= 5)
    {
        printf("\nResult Matrix:\n");
        for (int i = 0; i < N; i++)
        {
            for (int j = 0; j < N; j++)
                printf("%.1f ", h_C_gpu[i * N + j]);
            printf("\n");
        }
    }

    free(h_A);
    free(h_B);
    free(h_C_cpu);
    free(h_C_gpu);
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}


Overwriting matrix_add.cu


In [ ]:
!nvcc -arch=sm_75 matrix_add.cu -o matrix_add


In [ ]:
!./matrix_add


Enter matrix size (N): 4

CPU Time (ms): 0.0010
GPU Kernel Time (ms): 0.1195

Result Matrix:
12.0 12.0 12.0 12.0 
12.0 12.0 12.0 12.0 
12.0 12.0 12.0 12.0 
12.0 12.0 12.0 12.0 


In [ ]:
!./matrix_add


Enter matrix size (N): 8

CPU Time (ms): 0.0010
GPU Kernel Time (ms): 0.0988


In [ ]:
!./matrix_add

Enter matrix size (N): 12

CPU Time (ms): 0.0010
GPU Kernel Time (ms): 0.0997


In [ ]:
!./matrix_add

Enter matrix size (N): 16

CPU Time (ms): 0.0020
GPU Kernel Time (ms): 0.0860


In [ ]:
!./matrix_add

Enter matrix size (N): 20

CPU Time (ms): 0.0030
GPU Kernel Time (ms): 0.1166


In [ ]:
!./matrix_add


Enter matrix size (N): 30

CPU Time (ms): 0.0030
GPU Kernel Time (ms): 0.1611


In [ ]:
!./matrix_add

Enter matrix size (N): 40

CPU Time (ms): 0.0070
GPU Kernel Time (ms): 0.0932


In [ ]:
!./matrix_add


Enter matrix size (N): 50

CPU Time (ms): 0.0110
GPU Kernel Time (ms): 0.1106


In [ ]:
!./matrix_add

Enter matrix size (N): 80

CPU Time (ms): 0.0250
GPU Kernel Time (ms): 0.1060


In [ ]:
!./matrix_add

Enter matrix size (N): 100

CPU Time (ms): 0.0420
GPU Kernel Time (ms): 0.1281


In [ ]:
!./matrix_add

Enter matrix size (N): 200

CPU Time (ms): 0.1640
GPU Kernel Time (ms): 0.1187


In [ ]:
!./matrix_add

Enter matrix size (N): 150

CPU Time (ms): 0.0960
GPU Kernel Time (ms): 0.1511


In [ ]:
!./matrix_add

Enter matrix size (N): 180

CPU Time (ms): 0.1380
GPU Kernel Time (ms): 0.1017


In [ ]:
!./matrix_add


Enter matrix size (N): 170

CPU Time (ms): 0.1720
GPU Kernel Time (ms): 0.1320


In [ ]:
!./matrix_add

Enter matrix size (N): 160

CPU Time (ms): 0.1140
GPU Kernel Time (ms): 0.0979


In [ ]:
!./matrix_add

Enter matrix size (N): 150

CPU Time (ms): 0.0960
GPU Kernel Time (ms): 0.0905


Writing matrix_mul.cu


In [ ]:
%%writefile matrix_mul.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <time.h>

__global__ void matrixMul(float *A, float *B, float *C, int N)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N)
    {
        float sum = 0.0f;
        for (int k = 0; k < N; k++)
            sum += A[row * N + k] * B[k * N + col];
        C[row * N + col] = sum;
    }
}

int main()
{
    int N;
    float valA = 1.0f, valB = 2.0f;

    printf("Enter matrix size (N): ");
    scanf("%d", &N);

    int size = N * N * sizeof(float);

    float *h_A = (float *)malloc(size);
    float *h_B = (float *)malloc(size);
    float *h_C_cpu = (float *)malloc(size);
    float *h_C_gpu = (float *)malloc(size);

    for (int i = 0; i < N * N; i++)
    {
        h_A[i] = valA;
        h_B[i] = valB;
    }

    clock_t cpu_start = clock();
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++)
        {
            float sum = 0.0f;
            for (int k = 0; k < N; k++)
                sum += h_A[i * N + k] * h_B[k * N + j];
            h_C_cpu[i * N + j] = sum;
        }
    clock_t cpu_end = clock();

    float cpu_time = 1000.0f * (cpu_end - cpu_start) / CLOCKS_PER_SEC;

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dim3 block(16, 16);
    dim3 grid((N + block.x - 1) / block.x,
              (N + block.y - 1) / block.y);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    matrixMul<<<grid, block>>>(d_A, d_B, d_C, N);
    cudaEventRecord(stop);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
    {
        printf("CUDA error: %s\n", cudaGetErrorString(err));
        return 1;
    }

    cudaEventSynchronize(stop);

    float gpu_time;
    cudaEventElapsedTime(&gpu_time, start, stop);

    cudaMemcpy(h_C_gpu, d_C, size, cudaMemcpyDeviceToHost);

    printf("\nCPU Time (ms): %.4f\n", cpu_time);
    printf("GPU Kernel Time (ms): %.4f\n", gpu_time);

    if (N <= 5)
    {
        printf("\nResult Matrix (GPU):\n");
        for (int i = 0; i < N; i++)
        {
            for (int j = 0; j < N; j++)
                printf("%.1f ", h_C_gpu[i * N + j]);
            printf("\n");
        }
    }

    free(h_A); free(h_B); free(h_C_cpu); free(h_C_gpu);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    cudaEventDestroy(start); cudaEventDestroy(stop);

    return 0;
}


Writing matrix_mul.cu


In [ ]:
!nvcc -arch=sm_75 matrix_mul.cu -o matrix_mul


In [ ]:
!./matrix_mul


Enter matrix size (N): 10

CPU Time (ms): 0.0060
GPU Kernel Time (ms): 0.1408


In [ ]:
!./matrix_mul


Enter matrix size (N): 50

CPU Time (ms): 0.3890
GPU Kernel Time (ms): 0.0963


In [ ]:
!./matrix_mul


Enter matrix size (N): 20

CPU Time (ms): 0.0240
GPU Kernel Time (ms): 0.1045


In [ ]:
!./matrix_mul


Enter matrix size (N): 30

CPU Time (ms): 0.0790
GPU Kernel Time (ms): 0.1024


In [ ]:
!./matrix_mul


Enter matrix size (N): 40

CPU Time (ms): 0.1970
GPU Kernel Time (ms): 0.1257


In [ ]:
!./matrix_mul


Enter matrix size (N): 35

CPU Time (ms): 0.2190
GPU Kernel Time (ms): 0.1269


In [ ]:
!./matrix_mul


Enter matrix size (N): 30

CPU Time (ms): 0.0780
GPU Kernel Time (ms): 0.0982


In [ ]:
!./matrix_mul


Enter matrix size (N): 45

CPU Time (ms): 0.2690
GPU Kernel Time (ms): 0.1004
